# AI-Text Detection — First-Draft Pipeline (RAID only)

GSBS545 final project. Validates the full pipeline end-to-end before adding self-generated data / OOD eval.

**Tasks:** binary (human vs AI) and multiclass attribution (which of RAID's 11 generators).
**Models:** LogReg, LightGBM, Keras MLP.

Runs fine on CPU. Set **Runtime → Change runtime type → GPU** if you want the Keras steps faster (the LightGBM steps stay on CPU either way). Run cells top to bottom.

In [1]:
!pip install -q datasets lightgbm
!pip install -q "numpy<2.2" datasets lightgbm

In [2]:
import time
import numpy as np
import pandas as pd
from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report
import lightgbm as lgb

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy, SparseCategoricalCrossentropy

SEED = 42
N_SOURCES = 1000
MAX_FEATURES = 10000
SVD_DIMS = 300
np.random.seed(SEED); tf.random.set_seed(SEED)

_t0 = time.time()
def log(msg): print(f"[{time.time()-_t0:6.1f}s] {msg}", flush=True)

In [3]:
# 1. Load RAID (first run downloads ~several GB; cached after)
log("loading RAID...")
raw = load_dataset("liamdugan/raid", split="train").to_pandas()
raw = raw[raw["attack"] == "none"]
log(f"after dropping adversarial: {len(raw):,} rows, {raw.source_id.nunique():,} sources")

[   0.0s] loading RAID...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/11.8G [00:00<?, ?B/s]

extra.csv:   0%|          | 0.00/3.71G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating extra split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/24 [00:00<?, ?it/s]

[ 305.8s] after dropping adversarial: 467,985 rows, 13,371 sources


In [4]:
# 2. Subsample sources (group unit = source_id)
rng = np.random.default_rng(SEED)
sources = np.sort(raw["source_id"].unique()); rng.shuffle(sources)
keep = set(sources[:N_SOURCES])
df = raw[raw["source_id"].isin(keep)].copy()
log(f"subsample: {df['source_id'].nunique()} sources, {len(df):,} total rows")

[ 305.9s] subsample: 1000 sources, 35,000 total rows


In [5]:
# 3. Build task datasets
# Binary: 1 human + 1 random AI per source -> balanced 1:1
humans = df[df["model"] == "human"].drop_duplicates("source_id")
ai_one = (df[df["model"] != "human"]
            .groupby("source_id", group_keys=False)
            .sample(n=1, random_state=SEED))
df_bin = pd.concat([humans, ai_one], ignore_index=True)
df_bin["label"] = (df_bin["model"] != "human").astype(int)

# Multiclass: 1 random decoding per (source, model) for the 11 AI models
df_mc = (df[df["model"] != "human"]
           .groupby(["source_id", "model"], group_keys=False)
           .sample(n=1, random_state=SEED))

log(f"binary    : {len(df_bin):,} rows, balance = {df_bin['label'].value_counts().to_dict()}")
log(f"multiclass: {len(df_mc):,} rows, {df_mc['model'].nunique()} classes")

[ 306.8s] binary    : 2,000 rows, balance = {0: 1000, 1: 1000}
[ 306.8s] multiclass: 11,000 rows, 11 classes


In [6]:
# 4. Group split on source_id (no source crosses train/test)
def group_split(d, test_size=0.2):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=SEED)
    tr, te = next(gss.split(d, groups=d["source_id"]))
    return d.iloc[tr].reset_index(drop=True), d.iloc[te].reset_index(drop=True)

bin_tr, bin_te = group_split(df_bin)
mc_tr,  mc_te  = group_split(df_mc)
assert set(bin_tr["source_id"]) & set(bin_te["source_id"]) == set()
assert set(mc_tr["source_id"])  & set(mc_te["source_id"])  == set()
log("group split OK")

[ 306.8s] group split OK


In [7]:
# 5. Featurize (TF-IDF) + dense SVD reduction
def featurize(train_text, test_text):
    vec = TfidfVectorizer(ngram_range=(1, 2), max_features=MAX_FEATURES,
                          sublinear_tf=True, min_df=2)
    return vec.fit_transform(train_text), vec.transform(test_text)

log("vectorizing...")
Xb_tr, Xb_te = featurize(bin_tr["generation"], bin_te["generation"])
Xm_tr, Xm_te = featurize(mc_tr["generation"],  mc_te["generation"])
yb_tr, yb_te = bin_tr["label"].values, bin_te["label"].values

mc_labels = sorted(mc_tr["model"].unique())
label2id  = {l: i for i, l in enumerate(mc_labels)}
ym_tr = mc_tr["model"].map(label2id).values
ym_te = mc_te["model"].map(label2id).values

log("reducing to dense (SVD)...")
svd_b = TruncatedSVD(n_components=SVD_DIMS, random_state=SEED).fit(Xb_tr)
Xb_tr_d = svd_b.transform(Xb_tr).astype(np.float32)
Xb_te_d = svd_b.transform(Xb_te).astype(np.float32)

svd_m = TruncatedSVD(n_components=SVD_DIMS, random_state=SEED).fit(Xm_tr)
Xm_tr_d = svd_m.transform(Xm_tr).astype(np.float32)
Xm_te_d = svd_m.transform(Xm_te).astype(np.float32)
log("featurization done")

[ 306.8s] vectorizing...
[ 313.7s] reducing to dense (SVD)...
[ 322.3s] featurization done


## Binary — human vs AI

In [8]:
log("=== BINARY ===")
lr = LogisticRegression(max_iter=2000, C=1.0).fit(Xb_tr, yb_tr)
log(f"LogReg   | bal_acc = {balanced_accuracy_score(yb_te, lr.predict(Xb_te)):.4f}")

gbm = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31,
                         random_state=SEED, n_jobs=-1, force_col_wise=True,
                         verbose=-1).fit(Xb_tr, yb_tr)
log(f"LightGBM | bal_acc = {balanced_accuracy_score(yb_te, gbm.predict(Xb_te)):.4f}")

inp = Input(shape=(SVD_DIMS,))
h = Dense(128, activation="relu", kernel_regularizer=l2(1e-4))(inp)
h = Dropout(0.4)(h)
h = Dense(64, activation="relu", kernel_regularizer=l2(1e-4))(h)
out = Dense(1, activation="sigmoid")(h)
nn = Model(inp, out)
nn.compile(optimizer=Adam(1e-3), loss=BinaryCrossentropy(), metrics=["accuracy"])
es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
nn.fit(Xb_tr_d, yb_tr, validation_split=0.15, epochs=30, batch_size=128,
       callbacks=[es], verbose=0)
nn_pred = (nn.predict(Xb_te_d, verbose=0).ravel() > 0.5).astype(int)
log(f"Keras MLP| bal_acc = {balanced_accuracy_score(yb_te, nn_pred):.4f}")

[ 322.3s] === BINARY ===
[ 322.4s] LogReg   | bal_acc = 0.8000
[ 324.9s] LightGBM | bal_acc = 0.8125


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[ 333.9s] Keras MLP| bal_acc = 0.5000


## Multiclass — attribution across RAID's 11 generators

LightGBM trains on the dense 300-dim SVD features so the 11-class fit stays fast.

In [9]:
log("=== MULTICLASS (11 RAID generators) ===")
gbm = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                         random_state=SEED, n_jobs=-1, force_col_wise=True,
                         verbose=-1, objective="multiclass",
                         num_class=len(mc_labels)).fit(Xm_tr_d, ym_tr)
gbm_pred = gbm.predict(Xm_te_d)
log(f"LightGBM | bal_acc = {balanced_accuracy_score(ym_te, gbm_pred):.4f}")
print(classification_report(ym_te, gbm_pred, target_names=mc_labels, zero_division=0))

inp = Input(shape=(SVD_DIMS,))
h = Dense(256, activation="relu", kernel_regularizer=l2(1e-4))(inp)
h = Dropout(0.4)(h)
h = Dense(128, activation="relu", kernel_regularizer=l2(1e-4))(h)
out = Dense(len(mc_labels), activation="softmax")(h)
nn = Model(inp, out)
nn.compile(optimizer=Adam(1e-3),
           loss=SparseCategoricalCrossentropy(from_logits=False),
           metrics=["accuracy"])
es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
nn.fit(Xm_tr_d, ym_tr, validation_split=0.15, epochs=30, batch_size=128,
       callbacks=[es], verbose=0)
nn_pred = nn.predict(Xm_te_d, verbose=0).argmax(axis=1)
log(f"Keras MLP| bal_acc = {balanced_accuracy_score(ym_te, nn_pred):.4f}")
log("DONE")

[ 333.9s] === MULTICLASS (11 RAID generators) ===
[ 382.0s] LightGBM | bal_acc = 0.4809


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

     chatgpt       0.63      0.47      0.54       200
      cohere       0.35      0.43      0.39       200
 cohere-chat       0.41      0.36      0.38       200
        gpt2       0.40      0.67      0.50       200
        gpt3       0.52      0.44      0.48       200
        gpt4       0.59      0.68      0.63       200
  llama-chat       0.61      0.55      0.58       200
     mistral       0.41      0.33      0.37       200
mistral-chat       0.40      0.43      0.42       200
         mpt       0.68      0.48      0.56       200
    mpt-chat       0.47      0.45      0.46       200

    accuracy                           0.48      2200
   macro avg       0.50      0.48      0.48      2200
weighted avg       0.50      0.48      0.48      2200

[ 391.2s] Keras MLP| bal_acc = 0.4936
[ 391.2s] DONE
